In [1]:
pip install requests pandas xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 kB 3.7 MB/s eta 0:00:00


In [2]:
import requests
import pandas as pd

# Lista krajów do analizy
countries = {
    'AUT': 'Austria', 'BEL': 'Belgium', 'BGR': 'Bulgaria', 'HRV': 'Croatia',
    'CYP': 'Cyprus', 'CZE': 'Czech Republic', 'DNK': 'Denmark', 'EST': 'Estonia',
    'FIN': 'Finland', 'FRA': 'France', 'DEU': 'Germany', 'GRC': 'Greece',
    'HUN': 'Hungary', 'IRL': 'Ireland', 'ITA': 'Italy', 'LVA': 'Latvia',
    'LTU': 'Lithuania', 'LUX': 'Luxembourg', 'MLT': 'Malta', 'NLD': 'Netherlands',
    'POL': 'Poland', 'PRT': 'Portugal', 'ROU': 'Romania', 'SVK': 'Slovakia',
    'SVN': 'Slovenia', 'ESP': 'Spain', 'SWE': 'Sweden'
}

# Wskaźnik stopy bezrobocia (% całkowitej siły roboczej)
indicator = "SL.UEM.TOTL.ZS"
indicator_name = "Unemployment Rate (% of total labor force)"
start_year = 2004
end_year = 2024

# Funkcja do pobierania danych
def get_world_bank_data(countries, indicator, start_year, end_year):
    all_data = []

    for country_code, country_name in countries.items():
        url = f"https://api.worldbank.org/v2/country/{country_code}/indicator/{indicator}?date={start_year}:{end_year}&format=json"
        response = requests.get(url)

        if response.status_code == 200:
            data = response.json()
            if len(data) > 1 and data[1]:
                for entry in data[1]:
                    year = int(entry.get("date"))
                    value = entry.get("value")
                    all_data.append({"Year": year, "Country": country_name, "Unemployment Rate (%)": value})

    return pd.DataFrame(all_data)

# Pobranie danych
df = get_world_bank_data(countries, indicator, start_year, end_year)

# Pivotowanie danych (rok jako indeks, kraje jako kolumny)
df_pivot = df.pivot(index="Year", columns="Country", values="Unemployment Rate (%)")

# Obliczenie zmian procentowych rok do roku
df_pct_change = df_pivot.pct_change() * 100  # Procentowa zmiana rok do roku

# Tworzenie arkusza informacyjnego
info_data = {
    "Description": [
        "Source: World Bank Open Data",
        f"Indicator: {indicator_name}",
        "Definition: Unemployment refers to the share of the labor force that is without work but available for and seeking employment.",
        f"Years Covered: {start_year}-{end_year}",
        "Countries Covered: " + ", ".join(countries.values())
    ]
}

df_info = pd.DataFrame(info_data)

# Zapis do pliku Excel z trzema arkuszami
output_file = "world_bank_unemployment_rate.xlsx"
with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    df_info.to_excel(writer, sheet_name="0 - Info", index=False)
    df_pivot.to_excel(writer, sheet_name="Unemployment Rate (%)")
    df_pct_change.to_excel(writer, sheet_name="Yearly Change (%)")

print(f"Plik zapisano jako: {output_file}")

Plik zapisano jako: world_bank_unemployment_rate.xlsx
